<a href="https://colab.research.google.com/github/kun1gund3/SVD-XT-Studio-v1-Colab-Optimized-Diffusion/blob/main/SVD-XT%20Studio%20v1%20-%20Colab-Optimized%20Diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# 🎥 SVD-XT Studio v1 - Colab-Optimized Diffusion
# ================================================================

# ----------------------------
# 1. INSTALL
# ----------------------------
!pip install -q diffusers transformers accelerate safetensors opencv-python pyngrok gradio imageio

# ----------------------------
# 2. IMPORTS
# ----------------------------
import torch, gc, time
import gradio as gr
from pyngrok import ngrok
from huggingface_hub import login

from diffusers import StableVideoDiffusionPipeline
from diffusers import DDIMScheduler, EulerDiscreteScheduler, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video

# ----------------------------
# 3. CLEANUP
# ----------------------------
gc.collect()
torch.cuda.empty_cache()

# ----------------------------
# 4. TOKENS
# ----------------------------
HF_TOKEN = "DEIN_HF_TOKEN"
NGROK_TOKEN = "DEIN_NGROK_TOKEN"

login(token=HF_TOKEN)
ngrok.set_auth_token(NGROK_TOKEN)

# ----------------------------
# 5. LOAD MODEL
# ----------------------------
print("🚀 Loading Stable Video Diffusion...")

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16"
)

pipe.enable_model_cpu_offload()

# ----------------------------
# 6. SCHEDULER SWITCH
# ----------------------------
def set_scheduler(name):
    if name == "DDIM":
        pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    elif name == "Euler":
        pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)
    elif name == "DPM++":
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# ----------------------------
# 7. GENERATION FUNCTION (FIXED)
# ----------------------------
def generate_video(
    image,
    scheduler,
    motion_id,
    noise_aug,
    fps,
    steps,
    seed_val,
    num_frames,
    decode_chunk,
    width,
    height,
    batch_size,
    loop_video,
    vram_safe
):
    if image is None:
        return []

    set_scheduler(scheduler)

    outputs = []

    for i in range(int(batch_size)):
        torch.cuda.empty_cache()
        gc.collect()

        img = image.convert("RGB").resize((int(width), int(height)))

        seed = torch.seed() if seed_val == -1 else int(seed_val) + i
        generator = torch.Generator(device="cuda").manual_seed(seed)

        try:
            # ----------------------------
            # SAFE SVD CALL (NO EXTRA ARGS)
            # ----------------------------
            output = pipe(
                image=img,
                num_frames=int(num_frames),
                num_inference_steps=int(steps),
                decode_chunk_size=int(decode_chunk),
                motion_bucket_id=int(motion_id),
                noise_aug_strength=float(noise_aug),
                generator=generator
            )

            frames = output.frames[0]

            # Loop video option
            if loop_video:
                frames = frames + frames[::-1]

            timestamp = int(time.time())
            path = f"/content/video_{timestamp}_{i}.mp4"

            export_to_video(frames, path, fps=int(fps))

            outputs.append(path)

        except Exception as e:
            print(f"❌ Error: {e}")
            continue

    return outputs

# ----------------------------
# 8. GRADIO UI
# ----------------------------
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎥 SVD-XT Studio v1 - Colab-Optimized Diffusion")

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="pil", label="Input Image")

            scheduler = gr.Dropdown(
                ["Default", "DDIM", "Euler", "DPM++"],
                value="Default",
                label="Scheduler"
            )

            motion_id = gr.Slider(1, 255, 127, label="Motion")
            noise_aug = gr.Slider(0.0, 0.5, 0.02, label="Noise")
            fps = gr.Slider(5, 20, 7, label="FPS")
            steps = gr.Slider(10, 40, 20, label="Steps")
            seed = gr.Number(value=-1, label="Seed")

            num_frames = gr.Slider(8, 40, 25, label="Frames")
            decode_chunk = gr.Slider(1, 4, 1, label="Decode Chunk")

            width = gr.Dropdown([512, 768, 1024], value=1024, label="Width")
            height = gr.Dropdown([288, 432, 576], value=576, label="Height")

            batch_size = gr.Slider(1, 3, 1, label="Batch Size")
            loop_video = gr.Checkbox(label="Loop Video")
            vram_safe = gr.Checkbox(value=True, label="VRAM Safe (ignored internally but kept UI)")

            btn = gr.Button("🚀 Generate")

        with gr.Column():
            output = gr.Gallery(label="Generated Videos")

    btn.click(
        fn=generate_video,
        inputs=[
            input_img,
            scheduler,
            motion_id,
            noise_aug,
            fps,
            steps,
            seed,
            num_frames,
            decode_chunk,
            width,
            height,
            batch_size,
            loop_video,
            vram_safe
        ],
        outputs=output
    )

# ----------------------------
# 9. RUN SERVER
# ----------------------------
ngrok.kill()
url = ngrok.connect(7860)
print("🔗 Public URL:", url)

demo.launch(server_port=7860, debug=True)